In [ ]:
import mujoco
from IPython.display import clear_output
import math
import time
import jax
import jax.numpy as jp
from mujoco_playground._src.locomotion.pendulum_inverse_pendulum.environment import DoublePendulumEnv

import pyinstrument

In [ ]:

model = mujoco.MjModel.from_xml_path(
    "mujoco_playground/_src/locomotion/pendulum_inverse_pendulum/xmls/pendulum_inverse_pendulum.xml"
)
print(model.nq, model.nv, model.nu)  # should print your joint/actuator counts

In [ ]:

# 1. Create the environment



#env = DoublePendulumEnv(config_overrides={"task": "balance"})
import random


env = DoublePendulumEnv(config_overrides={"task": "swingup"})
print("✓ Environment created")
print(f"  action_size: {env.action_size}")
print(f"  xml_path: {env.xml_path}")
print(f"env._config.task: {env._config.task}")

# 2. Test reset
rng = jax.random.PRNGKey(random.randint(0, 2**32 - 1))
state = env.reset(rng)
print("\n✓ Reset successful")
print(f"  obs shape:    {state.obs.shape}")       # should be (6,)
print(f"  reward:       {state.reward}")          # should be 0.0
print(f"  done:         {state.done}")            # should be 0.0

# 3. Test a single step with random action
rng, action_rng = jax.random.split(rng)
action = jax.random.uniform(action_rng, (env.action_size,), minval=-1.0, maxval=1.0)
state = env.step(state, action)
print("\n✓ Step successful")
print(f"  obs shape:    {state.obs.shape}")       # should be (6,)
print(f"  reward:       {state.reward}")          # should be a float
print(f"  done:         {state.done}")            # should be 0.0 unless NaN

# 4. Test a full rollout
states = []
print("\nRunning 100 steps...")
for i in range(500):
    rng, action_rng = jax.random.split(rng)

    action_scale = 0.08
    #action = jax.random.uniform(action_rng, (env.action_size,), minval=-action_scale, maxval=action_scale)
    action = jax.numpy.ones(shape=(env.action_size,))*action_scale
    state = env.step(state, action)

    states.append(state)
    clear_output(wait=True)
    
    env.simple_render(state)
    print(f"Step {i+1}, reward: {state.reward:.4f}, angles: {state.obs[:4]},  {action} , metrics: {state.metrics['reward/continuity_cost']:.9f}     {state.data.ctrl}")

print(f"✓ 100 steps completed, mean reward: {sum([float(state.reward) for state in states])/len(states):.6f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt




sns.lineplot(x=range(len(states)), y = [float(state.obs[1]) for state in states] )
sns.lineplot(x=range(len(states)), y = [float(state.obs[2]) for state in states] )
plt.xlabel("Time Step")
plt.ylabel("Position Difference")
plt.title("Position Difference Over Time")
plt.legend()
plt.show()

sns.lineplot(x=range(len(states)), y=[float(state.reward) for state in states])
print("total reward:",sum([float(state.reward) for state in states]))

In [ ]:
#speed test:
import time

steps = 500


profiler = pyinstrument.Profiler()
profiler.start()

for _ in range(steps):
    rng, action_rng = jax.random.split(rng)
    action = jax.random.uniform(action_rng, (env.action_size,), minval=-1.0, maxval=1.0)
    
    

    state = env.step(state, action)


profiler.stop()
profiler.write_html("profile.html")

    
    
    

